## Welcome to Lab 3 Exercise

Evaluator + business assistant with lead capture

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Lamin"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Lamin"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Lamin"},
    {"role": "assistant", "content": "Well hi there, Lamin. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# And now - TOOLS!

Let's start with a function...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Step 1 - write some json to describe the tool


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
             
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

## Exercise 1: Add an evaluator LLM call

In [ ]:
# Step 1: define the evaluation format with structured outputs
from pydantic import BaseModel
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
# Step 2: Write the evaluator's system prompt
evaluator_system_prompt = f"""
You evaluate replies from an AI digital twin of Lamin Touray, chatting with visitors to his website.
Decide whether the latest reply is acceptable. It must:
1. Be strictly about Lamin's career, background, skills, experience or businesses. Personal topics
(food, sport, hobbies) are NOT acceptable.
2. Be accurate to the context below, with no invented facts.
3. Be professional and engaging.
If the visitor asked an off-topic question, a polite redirect back to professional topics IS acceptable.

## Context
{summary}

{linkedin}
"""

In [ ]:
# Step 3: Write the evaluation function
def evaluate(reply, message, history):
    user_prompt = f"""Conversation so far:
{history}

Latest visitor message:
{message}

Agent's reply:
{reply}

Is this reply acceptable? Give feedback either way."""
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    response = openai.chat.completions.parse(model="gpt-5.4-mini", messages=messages,
                                             response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
# Step 4: Pull the agent loop into its own function so you can reuse it on retries.
def run_agent_loop(messages):
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        for tool_call in msg.tool_calls:
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    return response.choices[0].message.content

In [ ]:
# Step 5: Update chat() to generate, evaluate and retry if necessary
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    user_msg = {"role": "user", "content": message}

    messages = [{"role": "system", "content": system_prompt}] + history + [user_msg]
    reply = run_agent_loop(messages)

    for attempt in range(2):  # cap retries
        evaluation = evaluate(reply, message, history)
        if evaluation.is_acceptable:
            print("Passed evaluation")
            return reply
        print(f"Rejected: {evaluation.feedback}")
        retry_prompt = system_prompt + f"""

## Your previous answer was rejected
Your reply: {reply}
Reason: {evaluation.feedback}
Try again, addressing this feedback."""
        messages = [{"role": "system", "content": retry_prompt}] + history + [user_msg]
        reply = run_agent_loop(messages)

    return reply

In [ ]:
# Step 6: Test it
gr.ChatInterface(chat).launch(inbrowser=True)

## Exercise 2: Apply it to your business

In [ ]:
# Step 1: Load the business context
with open("twin/business.txt", "r", encoding="utf-8") as f:
    business_context = f.read()

In [ ]:
# Let's display the business context so we can see what the agent has to work with
display(Markdown(business_context))

In [ ]:
# Step 2: Define the tools and the name-to-function lookup.
def record_user_details(email, name="Not provided", notes="Not provided"):
    print(f"Recorded lead: {email}, {name}, {notes}")
    with open("leads.txt", "a", encoding="utf-8") as f:
        f.write(f"{name} | {email} | {notes}\n")
    return "Details recorded successfully"

def record_unknown_question(question):
    print(f"Recorded unknown question: {question}")
    with open("unknown_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Question recorded successfully"

record_user_details_json = {
    "name": "record_user_details",
    "description": "Record that a visitor wants to get in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The visitor's email"},
            "name": {"type": "string", "description": "The visitor's name if given"},
            "notes": {"type": "string", "description": "What they're interested in"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Record a question that you couldn't answer from the context",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"}
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
] 

TOOL_FUNCTIONS = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unknown_question
}


In [ ]:
# Step 3: Rewrite the system prompt so the assistant represents my business rather than my personal life. 
system_prompt = f"""
# Your role
You are the AI assistant on the website of Lamin Touray, representing his two businesses:
Touray Solutions (web development) and Datafin (data consultancy).
You answer visitors' questions about the services, typical clients and past projects.
If asked, explain clearly that you are an AI assistant.

# Context
{business_context}

# Rules
- Be professional and engaging, as if talking to a potential client.
- Steer off-topic questions back to the businesses.
- If a visitor seems interested in working together, ask for their name and email, then call record_user_details with a short note about what they need.
- If you can't answer a question from the context, say so, and call record_unknown_question. Never make up an answer.
"""


In [ ]:
# Step 4: Point the evaluator at the business context
evaluator_system_prompt = f"""
You evaluate replies from the AI assistant on Lamin Touray's business website,
representing Touray Solutions (web development) and Datafin (data consultancy).
Decide whether the latest reply is acceptable. It must:
1. Be strictly about the businesses: services, clients, past projects or getting in touch. Personal topics are NOT acceptable.
2. Be accurate to the context below, with no invented facts (e.g. no made-up prices).
3. Be professional and engaging.
A polite redirect of an off-topic question, or saying "I don't know", IS acceptable.

## Context
{business_context}
"""

In [ ]:
# Step 5: Agent loop with generic tool dispatch.
def run_agent_loop(messages):
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        msg = response.choices[0].message
        messages.append(msg)
        for tool_call in msg.tool_calls:
            func = TOOL_FUNCTIONS.get(tool_call.function.name)
            args = json.loads(tool_call.function.arguments)
            result = func(**args) if func else "Unknown tool"
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)  
    return response.choices[0].message.content

In [ ]:
# Step 6: chat() - same as Exercise 1, now using the business prompt.
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    user_msg = {"role": "user", "content": message}

    messages = [{"role": "system", "content":system_prompt}] + history + [user_msg]
    reply = run_agent_loop(messages)

    for attempt in range(2):  # cap retries
        evaluation = evaluate(reply, message, history)
        if evaluation.is_acceptable:
            print("Passed evaluation")
            return reply
        print(f"Rejected: {evaluation.feedback}")
        retry_prompt = system_prompt + f"""
## Your previous answer was rejected
Your reply: {reply}
Reason: {evaluation.feedback}
Try again, addressing this feedback."""
        messages = [{"role": "system", "content": retry_prompt}] + history + [user_msg]
        reply = run_agent_loop(messages)

    return reply

### Let's test these in the chat; 
1. I’d like a website for my café, I’m Sam, sam@test.com 
2. How much do you charge?

In [ ]:
# Step 7: Test it
gr.ChatInterface(chat).launch(inbrowser=True)